# Embeddings

<img src="images/embeddings_intro_vignette.jpg" width="150" alt="Embeddings introduction" style="float: left; margin-right: 15px; margin-bottom: 10px;">

In Part 3 we opened up the first black box: what a *token* is, and how a tokenizer turns text into a sequence of token ids. But a token id is just an integer — index 2644, say. On its own, an integer tells the model nothing about what that token *means*, or how it relates to any other token. In this part we open up the next black box: how a model turns each token id into a rich, continuous vector called an **embedding**, and what we can do with these vectors once we have them.

## What is an embedding?

<br>
<img src="images/embeddings_academic.jpg" width="450" alt="Vector Embedding Space Diagram" style="display: block; margin: 15px auto;">
<br>


<img src="images/embeddings_vignette.jpg" width="150" alt="What is an embedding" style="float: left; margin-right: 15px; margin-bottom: 10px;">

An **embedding** represents a token (or a larger piece of text) as a point in an *n*-dimensional space: a vector of *n* real-number coordinates, one per dimension — *n* is typically a few hundred to a few thousand (576, for the small model we'll use below). Concretely, this vector is stored as one row of a matrix inside the model, one row per token in the vocabulary: row *i* gives the *n* coordinates that locate token *i* in that space.

These coordinates are *learned* during training: the model positions each token's point so that tokens which behave similarly in text end up nearby, and tokens that behave differently end up far apart. This is what lets a model generalize: two tokens whose points are close together in this space can be treated in comparable ways by the rest of the model — because training placed them there based on how they're actually used across huge amounts of text, not based on how they're spelled or what they "mean" to us.

## Extracting real embeddings from a model

<img src="images/extraction_vignette.jpg" width="150" alt="Extracting real embeddings" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Let's look at actual embeddings inside a real model: [`HuggingFaceTB/SmolLM2-135M`](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), the same small local model we used in Part 3 for Program 8. Every model built with `transformers` exposes its embedding matrix through `get_input_embeddings()`.

In [ ]:
# Program 1: inspecting a model's embedding matrix

from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForCausalLM

load_dotenv(override=True)  # picks up HF_TOKEN from .env, if you set one back in Part 3

model_name = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# The embedding matrix: one row per token in the vocabulary, one column per embedding dimension.
embedding_matrix = model.get_input_embeddings().weight
vocab_size, embedding_dim = embedding_matrix.shape
print(f"Embedding matrix shape: {vocab_size} tokens x {embedding_dim} dimensions")

# Encode a single word (note the leading space, exactly like in Part 3: it's part of the token).
token_id = tokenizer.encode(" cat", add_special_tokens=False)[0]
print(f"Token id for ' cat': {token_id}")

# The embedding of this token is simply the row of the matrix at that index:
# a single vector, not a matrix, so its "shape" has only one number in it -
# the number of coordinates it has (embedding_dim, the same 576 as above).
cat_vector = embedding_matrix[token_id]
print(f"Vector shape: {cat_vector.shape} -- a single vector of {cat_vector.shape[0]} coordinates")
print(f"First 8 (of {cat_vector.shape[0]}) values: {cat_vector[:8].tolist()}")

576 numbers, on their own, don't mean much just by looking at them. What makes embeddings useful is *comparing* them to one another.

## The distance between tokens

<br>
<img src="images/cosine_similarity_academic.jpg" width="400" alt="Cosine Similarity Diagram" style="display: block; margin: 15px auto;">
<br>

<img src="images/distance_vignette.jpg" width="150" alt="The distance between tokens" style="float: left; margin-right: 15px; margin-bottom: 10px;">

If similar tokens end up with similar vectors, we should be able to start from one token's embedding and find its nearest neighbors in the whole embedding matrix — the tokens the model considers "closest" to it. We'll measure closeness with **cosine similarity** (how aligned two vectors are, from -1 to 1, ignoring their length) rather than raw distance, since it's the standard choice for comparing embeddings.

In [ ]:
# Program 2: finding the nearest-neighbor tokens by cosine similarity

import torch
import torch.nn.functional as F

# Note: this reuses `embedding_matrix` and `tokenizer` from Program 1 above (Jupyter cells
# share the same variables). We don't need `model` itself again here -- everything we need
# from it (the embedding matrix) was already extracted into `embedding_matrix` in Program 1.

def nearest_tokens(word, k=8):
    """Print the k tokens whose embedding is most similar to `word`'s."""
    token_id = tokenizer.encode(word, add_special_tokens=False)[0]
    vector = embedding_matrix[token_id]  # shape: (embedding_dim,) -- a single vector

    # F.cosine_similarity compares two tensors of the same shape, element-wise pair by pair.
    # `vector` alone is 1D (embedding_dim,), so unsqueeze(0) turns it into (1, embedding_dim):
    # a "batch" of a single vector. Compared against embedding_matrix (vocab_size, embedding_dim),
    # this one row automatically gets compared against every row of the matrix (broadcasting).
    # dim=1 tells it to compute the similarity along the embedding_dim axis (each row),
    # so the result `similarities` is a 1D tensor of shape (vocab_size,): one score per
    # token in the vocabulary, telling us how aligned its vector is with `vector`.
    similarities = F.cosine_similarity(vector.unsqueeze(0), embedding_matrix, dim=1)

    # torch.topk(similarities, k) scans that whole (vocab_size,) tensor and returns the k
    # largest values directly, already sorted from most to least similar. It gives back a
    # named tuple: `.values` (the k similarity scores themselves) and `.indices` (their
    # positions in the tensor -- which, here, are exactly the corresponding token ids).
    top = torch.topk(similarities, k)
    for score, neighbor_id in zip(top.values.tolist(), top.indices.tolist()):
        print(f"{score:.3f}  {neighbor_id:>6}  {tokenizer.decode([neighbor_id])!r}")

print("--- nearest to ' cat' ---")
nearest_tokens(" cat")
print("\n--- nearest to ' king' ---")
nearest_tokens(" king")

This is a genuinely striking result: without ever telling the model what a cat or a king *is*, training alone placed " cat" next to its capitalized and plural forms, and right next to " dog" — a semantically related animal. " king" ends up close to " King", " kings", but also " queen", " Queen", " emperor", " prince" — related roles of power, not just spelling variants. This is what an embedding space captures: not visual or spelling similarity, but *how a token is used*, learned purely from huge amounts of text.

Your turn to play: try `nearest_tokens(" paris")` or `nearest_tokens(" happy")` above and see what the model considers close.

In [ ]:
# write tour code with the nearest token to " paris", " happy" and " orange".

## Relationships between tokens: word analogies

<img src="images/analogies_vignette.jpg" width="150" alt="Word analogies" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Nearest neighbors show which tokens are alike, but embedding spaces often encode something richer: *relationships*, as a consistent direction. The classic example is gender: the shift from "man" to "woman" should look a lot like the shift from "uncle" to "aunt", or from "king" to "queen" — same relationship, applied to different pairs.

<br>
<img src="images/cosine_similarity_academic.jpg" width="200" alt="Cosine Similarity Diagram" style="display: block; margin: 15px auto;">
<br>

As a reminder, **cosine similarity** measures how aligned two vectors are, from -1 (opposite directions) to 1 (same direction), with 0 meaning unrelated — that's what every `cosine=...` value below refers to. Program 3 introduces a second way to measure distance, **Euclidean distance**, alongside it.

For this, we'll switch away from `SmolLM2`'s embeddings and load real, classic **GloVe** word vectors instead (via [`gensim`](https://radimrehurek.com/gensim/)) — pretrained specifically to make this kind of linear relationship come out cleanly, unlike `SmolLM2`'s embeddings, which are just a by-product of training a full causal LLM and were never optimized for this. This is a lightweight download (~130MB, no GPU needed), so it runs comfortably on a laptop.

> *[`gensim`](https://radimrehurek.com/gensim/) is a Python library for classic, pre-transformer **NLP** (Natural Language Processing): static word vectors (word2vec, GloVe, FastText) and topic modelling (LDA, LSA), with no deep learning framework or GPU involved. Its `gensim.downloader` module — what we'll use below — gives one-line access to a small, curated catalog of well-known pretrained vector sets and benchmark corpora, GloVe among them. That's a much narrower job than the Hugging Face Hub we've used elsewhere in this course (Part 3, Program 8): Hugging Face hosts hundreds of thousands of community-contributed models across every modality (text, vision, audio...), mostly modern transformer-based ones producing contextual embeddings — the same word gets a different vector depending on its sentence, unlike GloVe's fixed one-vector-per-word. Reach for `gensim` specifically for static word vectors or topic models; reach for Hugging Face for essentially everything trained since transformers took over, including the contextual embeddings we extracted from `SmolLM2` earlier in this notebook.*

In [ ]:
# Program 3: distance within pairs, and between the pairs' relationships (GloVe)

import gensim.downloader as gensim_api
import numpy as np

# Loads (and caches locally after the first run) 400,000 GloVe word vectors, 100
# dimensions each -- trained on Wikipedia + Gigaword, completely unrelated to SmolLM2.
glove = gensim_api.load("glove-wiki-gigaword-100")
print(f"GloVe vocabulary: {len(glove)} words, {glove.vector_size} dimensions each")

# No leading-space convention here: GloVe was trained on plain, lowercased words.
gender_pairs = [("man", "woman"), ("uncle", "aunt"), ("king", "queen"), ("dad", "mom")]
contrast_pairs = [("love", "hate"), ("caribou", "bread")]

def euclidean(a, b):
    """Euclidean distance: the ordinary straight-line distance between two points a and b in
    space, here the two words' embedding vectors -- computed as the norm (length) of their
    difference. Unlike cosine similarity, it does depend on the vectors' length, not just
    their direction."""
    return float(np.linalg.norm(glove[a] - glove[b]))

print("--- how close is each pair to itself? ---")
for a, b in gender_pairs + contrast_pairs:
    print(f"{a:10} <-> {b:10}  cosine={glove.similarity(a, b):.3f}  euclidean={euclidean(a, b):.2f}")

# Same idea as before: the "relationship" between two words is the difference between
# their vectors, always subtracted in the same order (male -> female here).
print("\n--- are the four gender pairs' relationships similar to each other? ---")
relationships = {f"{a} -> {b}": glove[b] - glove[a] for a, b in gender_pairs}
labels = list(relationships.keys())
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        similarity = glove.cosine_similarities(relationships[labels[i]], [relationships[labels[j]]])[0]
        print(f"{labels[i]!r:20} vs {labels[j]!r:20}  cosine={similarity:.3f}")

# Same negative control as before: "love -> hate" isn't a gender relationship.
print("\n--- for contrast: is 'love -> hate' aligned with the gender direction? ---")
love_to_hate = glove["hate"] - glove["love"]
for label, relationship in relationships.items():
    similarity = glove.cosine_similarities(relationship, [love_to_hate])[0]
    print(f"{label!r:20} vs 'love -> hate'      cosine={similarity:.3f}")

This time the structure is much cleaner. The four gender pairs' relationship vectors now agree with each other in the 0.45-0.59 range — a clear, consistent signal that a shared "gender" direction really is there.

The negative control is clean too: `"hate" - "love"` scores **close to zero or slightly negative** (-0.13 to -0.04) against all four gender vectors — a sharp "not aligned" signal. And in the first table, "caribou"/"bread" (two unrelated words) drops to cosine 0.17 — GloVe pushes truly unrelated words apart, rather than leaving everything mildly positive.

Same underlying idea as before (a relationship is a direction, and direction only makes sense if you're consistent about which end you subtract from which) — just measured with vectors actually trained to make this kind of linear structure emerge, instead of a causal LLM's raw input embeddings.

## Averaging pairs to isolate the relationship

<img src="images/averaging_vignette.jpg" width="150" alt="Averaging pairs" style="float: left; margin-right: 15px; margin-bottom: 10px;">

A single pair's difference vector (like `"woman" - "man"`) doesn't capture *only* gender — it also carries whatever else happens to distinguish that specific pair (word frequency, other connotations...), which is why two individual gender pairs only agreed at 0.45-0.59 above, not closer to 1. The standard fix (used, for example, by [Bolukbasi et al.](https://arxiv.org/abs/1607.06520) to study gender bias in embeddings) is to **average several pairs' difference vectors together**: the shared "gender" component reinforces itself across pairs, while each pair's own idiosyncratic noise, pointing in different directions, tends to cancel out.

Let's try it with eight gender pairs instead of four, and — to avoid the obviously unfair trick of comparing a pair to an average that already includes itself — check each pair against the average of *all the others only*.

In [ ]:
# Program 4: averaging several pairs to cancel out per-pair noise

more_gender_pairs = [
    ("man", "woman"), ("uncle", "aunt"), ("king", "queen"), ("dad", "mom"),
    ("brother", "sister"), ("husband", "wife"), ("actor", "actress"), ("waiter", "waitress"),
]
all_relationships = [glove[b] - glove[a] for a, b in more_gender_pairs]

print("--- each pair vs. the average of the seven OTHER pairs (no self-comparison) ---")
for i, (a, b) in enumerate(more_gender_pairs):
    # This average is recomputed on every iteration because it's a different set each
    # time: "every relationship except the current one" (leave-one-out), so pair i is
    # never compared against an average that already includes itself.
    other_relationships = [rel for j, rel in enumerate(all_relationships) if j != i]
    average_direction = np.mean(other_relationships, axis=0)  # cancels out each pair's own noise
    similarity = glove.cosine_similarities(all_relationships[i], [average_direction])[0]
    print(f"{a}->{b:10} vs average-of-others   cosine={similarity:.3f}")

# Now average ALL eight pairs together: this is our best estimate of "the" gender direction.
full_gender_direction = np.mean(all_relationships, axis=0)

print("\n--- do our earlier contrast pairs align with this averaged gender direction? ---")
love_to_hate = glove["hate"] - glove["love"]
caribou_to_bread = glove["bread"] - glove["caribou"]
print(f"love->hate         vs gender direction   cosine={glove.cosine_similarities(love_to_hate, [full_gender_direction])[0]:.3f}")
print(f"caribou->bread     vs gender direction   cosine={glove.cosine_similarities(caribou_to_bread, [full_gender_direction])[0]:.3f}")

Clearly better: each pair now agrees with the average of the *other seven* at 0.34 to 0.82 (most well above 0.6), compared to 0.45-0.59 for single-pair-vs-single-pair comparisons earlier — averaging really does cancel out per-pair noise and isolate a cleaner shared direction (`"husband"->"wife"` is the noisiest of the eight at 0.34, `"actor"->"actress"` the cleanest at 0.82).

And the contrast pairs now separate much more sharply: `"love"->"hate"` scores **clearly negative** against this averaged direction (rather than the mild -0.13 to -0.04 we saw pair-by-pair), and `"caribou"->"bread"` lands almost exactly at **0** — about as close to "no relationship at all" as cosine similarity gets. Averaging didn't just make the *positive* signal cleaner; it made the *absence* of a gender relationship in these two unrelated pairs unambiguous too.

In [ ]:
# Program 5: the classic analogy, as vector arithmetic -- king - man + woman ~= ? (GloVe)

# gensim's most_similar() does exactly this arithmetic for us: add the "positive" vectors,
# subtract the "negative" ones, then find the closest words to the result (excluding the
# words used to build it).
top_matches = glove.most_similar(positive=["king", "woman"], negative=["man"], topn=8)

print("'king' - 'man' + 'woman' ~= ?\n")
for word, score in top_matches:
    print(f"{score:.3f}  {word!r}")

And there it is: "queen" comes out on top with 0.770, well clear of the next candidate ("monarch" at 0.684) — "king" doesn't even make the list (`most_similar` automatically excludes the words used to build the query). The rest of the list reads like a plausible dictionary of royalty-and-gender-adjacent words too: throne, daughter, princess, prince, elizabeth, mother.

Nobody told GloVe that a queen is to a king what a woman is to a man — this falls directly out of doing arithmetic on vectors that were only ever trained to predict which words tend to appear near each other in a huge text corpus. This is the clearest illustration yet that an embedding space isn't just a way to measure similarity: it encodes *structure*, well-behaved enough that relationships between concepts can be manipulated like ordinary vectors — and the cleaner the training objective is *for this specific purpose*, the more cleanly that structure shows up.

# Inside the network: architecture and training

<img src="images/network_inside_vignette.jpg" width="150" alt="Inside the network" style="float: left; margin-right: 15px; margin-bottom: 10px;">

In Part 3 we saw a real model, `SmolLM2`, resolve one token at a time from raw logits (Program 8). Earlier in this notebook we saw that each token id is really just a row of a big **embedding matrix**. But we skipped over what happens in between: a matrix of numbers goes in, and a single token comes out — what's actually inside that box, and why does it produce a *useful* token rather than a random one?

This part opens that box: the generic architecture shared by word2vec-style networks and modern LLMs alike, and the training process that turns a network's initially-random weights into something that predicts well.

*Main source for this part: Google's [Machine Learning Crash Course, "Obtaining embeddings"](https://developers.google.com/machine-learning/crash-course/embeddings/obtaining-embeddings) (video: [youtu.be/my5wFNQpFO0](https://youtu.be/my5wFNQpFO0)), which walks through the same word2vec architecture we build intuition for below.*

## Anatomy of a tiny neural network

A neural network trained to produce embeddings is built from a stack of **layers**, each one a set of *neurons* connected to the layer before it by weighted links:

* **Input layer**: one neuron per word in the vocabulary, using a *one-hot* encoding — the neuron for the current word is set to 1, every other neuron is 0.
* **Projection (embedding) layer**: fully connected to the input layer, with one neuron per embedding dimension (10, in a toy example; 576 for `SmolLM2`, as we saw above). No activation function here — this layer's job is purely to project the huge one-hot input down to a much smaller, dense vector. Its weights, once trained, *are* the embedding matrix from earlier in this notebook.
* **Hidden layer(s)**: fully connected, this time *with* an activation function, to let the network combine and further process that dense vector.
* **Output layer**: back to one neuron per vocabulary word, passed through a **softmax** function so the outputs form a probability distribution (they sum to 1) — exactly the `torch.softmax` step we used by hand in Part 3, Program 8.

<br>
<img src="images/neural_network_architecture_academic.jpg" width="500" alt="Neural Network Architecture Diagram" style="display: block; margin: 15px auto;">
<br>

## From matrix to token: this is exactly what Part 3, Program 8 did

This architecture isn't an abstract exercise — it's precisely what ran, layer by layer, every time we called `local_model(generated_ids)` in Part 3's Program 8:

1. Each input token id selects one row of the embedding matrix — the *projection layer* above, already trained.
2. That vector flows through many hidden layers (a lot more, and far more sophisticated, than our one hidden layer here — this is where `SmolLM2`'s actual "reasoning" happens, a black box we won't open in this course).
3. The final layer produces one raw score (a *logit*) per vocabulary token.
4. `torch.softmax` turns those logits into a probability distribution — our output layer above.
5. Greedy decoding (`torch.argmax`) just reads off the most likely token.

In other words: "a matrix goes in, a token comes out" *is* a forward pass through a network built exactly like this — deeper and wider, but the same shape.

## How training assigns the weights

<img src="images/training_feedback_soviet.jpg" width="150" alt="Training weights feedback" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Right after it's built, a network's weights are random: feed it a word, and its "prediction" is noise. Training is the process that turns those random weights into useful ones, by repeating a simple loop, millions or billions of times, over huge amounts of text:

1. **Predict**: run the current word (or sequence) through the network, forward, exactly as above.
2. **Compare**: check that prediction against the word that *actually* came next (or nearby) in the real text — the further off the prediction, the higher this **loss**.
3. **Adjust**: nudge every weight in the network a tiny bit, in whichever direction would have reduced that error — this step is called **backpropagation**, and the nudging rule is **gradient descent**.
4. **Repeat**, on the next example.

Nothing here is specific to word2vec or to language: it's the same three-step loop that trains any neural network. What makes it work for embeddings specifically is *what* we ask the network to predict — a nearby word, as we saw earlier in this course.

We've actually already run this exact loop ourselves, in [`Attic_word2vec_training.ipynb`](Attic_word2vec_training.ipynb)'s skip-gram training program: `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` *are*, literally, steps 3 (backpropagation) and 4 (the weight update) above, and `F.cross_entropy(...)` computes the loss of step 2 — just at a toy scale (14 words, 1500 steps, seconds on a laptop), instead of a real LLM's scale (tens of thousands of vocabulary tokens, trillions of training tokens, weeks on a cluster of GPUs).

<br>
<img src="images/training_loop_academic.jpg" width="450" alt="Training Loop Diagram" style="display: block; margin: 15px auto;">
<br>

## What's in a model name? Counting the weights

<img src="images/weights_vignette.jpg" width="150" alt="Counting parameters" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Every weighted link between neurons in this architecture is a **parameter** the network has to learn — and a model's name usually tells you how many of them it has: the number just before "M" (million) or "B" (billion) is its total parameter count, summed across every layer (the projection layer above, plus every hidden layer we won't open in this course).

| Model | Parameter count |
|---|---|
| `SmolLM2-135M` (the model we've used since Part 3) | 135 million |
| `SmolLM2-1.7B` (its largest sibling in the same family) | 1.7 billion |
| Mistral 7B (the `mistral` models we've queried since Part 1) | ~7.3 billion |
| Llama 3 70B | 70 billion |

Same family, same architecture shape — just more (and wider) layers as the parameter count grows.

## Same weights, fewer bytes: quantization

| Precision | Bytes per weight | Size of a 7-billion-parameter model |
|---|---|---|
| FP32 (full precision) | 4 | ~28 GB |
| FP16 / BF16 (half precision) | 2 | ~14 GB |
| INT8 | 1 | ~7 GB |
| INT4 | 0.5 | ~3.5 GB |

Reducing precision like this is called **quantization**, and it's exactly why `ollama pull mistral` (Part 1) only downloads a few gigabytes, not 28 — despite Mistral being a 7-billion-parameter model, Ollama's default download is quantized (commonly to 4 bits), trading a bit of precision for a file that actually fits, and runs, on a laptop.

## Your turn to calculate (no code needed)

Back in Program 1, we printed: `Embedding matrix shape: 49152 tokens x 576 dimensions`.

1. How many individual weights does that projection layer alone contain?
2. `SmolLM2-135M` has 135 million parameters in total. What share of the *whole model* is just this one layer?
3. How many megabytes would this layer alone take up at FP32 precision, versus INT4?

Work it out, then check below.

> **Answer:** 49,152 × 576 = 28,311,552 weights — about **21%** of `SmolLM2-135M`'s 135 million parameters, all in this one lookup table. At FP32 (4 bytes/weight) that's ~113 MB; at INT4 (0.5 byte/weight) it shrinks to ~14 MB — over a fifth of the whole model, just for looking up which row of a matrix a token id points to.

## Key takeaways

<img src="images/takeaways.jpg" width="150" alt="Key takeaways" style="float: left; margin-right: 15px; margin-bottom: 10px;">

* A network that produces embeddings is a stack of layers: a one-hot input, a projection/embedding layer (no activation — its weights *are* the embedding matrix), one or more hidden layers, and a softmax output.
* "A matrix goes in, a token comes out" is exactly this stack's forward pass — the same mechanism we ran by hand in Part 3, Program 8, and the same mechanism behind any modern LLM, just far deeper and wider.
* Training is what turns random weights into useful ones: predict, compare to the real answer (loss), adjust every weight to reduce that error (backpropagation / gradient descent), repeat — the exact three lines of code (`zero_grad`, `backward`, `step`) we already ran ourselves in `Attic_word2vec_training.ipynb`'s toy word2vec.
* A model's name usually encodes its total parameter count (135M, 7B, 70B...); how many bytes each of those weights takes (its precision) sets its real size — quantization (FP32 → INT4) is what lets a multi-billion-parameter model fit, and run, on a laptop.
* Once training stops, the weights are frozen: what the network "knows" is exactly what those weights encode, from whatever data it was trained on — a limitation we'll run into directly in Part 5.

## Function reference

A quick reference for the less obvious functions and methods used in this notebook (skipping ones already familiar from earlier parts, like `AutoTokenizer.from_pretrained`):

| Function (module) | Arguments | Returns | Used in |
|---|---|---|---|
| `torch.topk(tensor, k)` (`torch`) | a tensor, `k` (int) | the `k` largest values and their indices, sorted descending (`.values`, `.indices`) | Program 2 |
| `F.cosine_similarity(a, b, dim=)` (`torch.nn.functional`) | two tensors (broadcastable), `dim` | a tensor of cosine similarities computed along `dim` | Program 2 |
| `gensim_api.load(name)` (`gensim.downloader`) | a model/corpus name (str), e.g. `"glove-wiki-gigaword-100"` | a `KeyedVectors` object holding the loaded word vectors | Program 3 |
| `np.linalg.norm(vector)` (`numpy`) | a vector/array | its length (Euclidean norm), as a float | Program 3 |
| `KeyedVectors.similarity(word1, word2)` (`gensim`) | two words (str) | the cosine similarity between their vectors (float) | Program 3 |
| `KeyedVectors.cosine_similarities(vector, vectors_all)` (`gensim`) | one vector, a list/array of vectors | an array of similarities, one per vector in `vectors_all` | Programs 3, 4 |
| `KeyedVectors.most_similar(positive=, negative=, topn=)` (`gensim`) | words to add/subtract, number of results | a list of `(word, score)` tuples, sorted by similarity | Program 5 |